# Folder format

> A directory of `.npz` columns plus one image file per step (and per agent) for image columns (stable-worldmodel's `folder` format).

In [ ]:
#| default_exp data.folder

In [ ]:
#| hide
from nbdev.showdoc import *

Layout (stable-worldmodel's, with per-agent image columns)::

    <root>/
      ep_len.npz, ep_offset.npz
      <col>.npz                                   # tabular columns
      <img_col>/ep_<i>_step_<j>.jpeg              # an image column, (H, W, C) per step
      <img_col>/ep_<i>_step_<j>_agent_<a>.jpeg    # a per-agent image column, (num_agents, H, W, C) per step
      metadata.json                               # the writer's metadata (env name, kwargs, ...)

A column is written as images when its steps are ``uint8`` arrays ``(H, W, 1|3)`` or, per agent,
``(num_agents, H, W, 1|3)`` (e.g. ``pixels``, ``goal``, ``render``); everything else goes to `.npz`.
Images are JPEG (lossy); per-agent image columns are read back as ``(T, num_agents, H, W, C)``,
single images channel-first ``(T, C, H, W)``, as in `HDF5Dataset`. Episode-scoped data is not
stored (dropped with a warning).

In [ ]:
#| export
# Adapted from stable-worldmodel's data package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import json
import logging
import re
import shutil
from pathlib import Path
from typing import Any, Callable

import numpy as np

from stable_marl.data.dataset import Dataset, get_cache_dir, to_tensor
from stable_marl.data.format import Format, register_format, split_episode_data, validate_write_mode

In [ ]:
#| export
def image_layout(vals) -> str | None:
    """
    How a column's steps are images: ``'single'`` (``uint8`` ``(H, W, 1|3)``), ``'agents'``
    (``uint8`` ``(num_agents, H, W, 1|3)``), or None (not images).
    """
    if not len(vals):
        return None
    sample = np.asarray(vals[0])
    if sample.dtype != np.uint8 or sample.shape[-1] not in (1, 3):
        return None
    return {3: 'single', 4: 'agents'}.get(sample.ndim)


def is_image_column(vals) -> bool:
    "Whether `vals` looks like a sequence of image frames (single or per agent)."
    return image_layout(vals) is not None


_AGENT_FILE = re.compile(r'ep_(\d+)_step_(\d+)_agent_(\d+)\.(jpe?g|png)$')


def _agents_of(col_dir: Path) -> int | None:
    "Number of agents of a per-agent image column (None for a single-image column)."
    agents = {int(m.group(3)) for p in col_dir.iterdir() if (m := _AGENT_FILE.match(p.name))}
    return max(agents) + 1 if agents else None


def _read_image(path: Path) -> np.ndarray:
    from PIL import Image
    for suffix in ('.jpeg', '.jpg', '.png'):
        if path.with_suffix(suffix).exists():
            return np.array(Image.open(path.with_suffix(suffix)))
    raise FileNotFoundError(f"no image file for {path}")


def _write_image(arr: np.ndarray, path: Path):
    from PIL import Image
    Image.fromarray(arr.squeeze(-1) if arr.shape[-1] == 1 else arr).save(path)

In [ ]:
#| export
class FolderDataset(Dataset):
    """
    Dataset of a folder (see above): `.npz` columns loaded into memory, images read per step.

    Parameters
    ----------
    name : str
        Dataset name, read from ``<cache dir>/datasets/<name>`` (or pass `path`)
    frameskip, num_steps, transform
        See :class:`Dataset`
    keys_to_load : list of str
        Columns to load (default: all)
    folder_keys : list of str
        Image columns (default: the subdirectories)
    """
    def __init__(
        self,
        name: str | None = None,
        frameskip: int = 1,
        num_steps: int = 1,
        transform: Callable[[dict], dict] | None = None,
        keys_to_load: list[str] | None = None,
        folder_keys: list[str] | None = None,
        cache_dir: str | Path | None = None,
        path: str | Path | None = None):
        if path is not None:
            self.path = Path(path)
        else:
            if name is None:
                raise TypeError("FolderDataset requires either `name` or `path`")
            self.path = Path(cache_dir or get_cache_dir(sub_folder='datasets')) / name
        if folder_keys is None:                       # every subdirectory is an image column
            folder_keys = [p.name for p in self.path.iterdir() if p.is_dir()]
        self.folder_keys = folder_keys
        self._agents = {k: _agents_of(self.path / k) for k in folder_keys if (self.path / k).is_dir()}
        self._cache: dict[str, np.ndarray] = {}
        meta = self.path / 'metadata.json'
        self.metadata = json.loads(meta.read_text()) if meta.exists() else {}

        lengths = np.load(self.path / 'ep_len.npz')['arr_0']
        offsets = np.load(self.path / 'ep_offset.npz')['arr_0']
        if keys_to_load is None:
            keys_to_load = sorted(p.stem if p.suffix == '.npz' else p.name for p in self.path.iterdir()
                                  if p.stem not in ('ep_len', 'ep_offset') and p.name != 'metadata.json')
        self._keys = keys_to_load
        for key in self._keys:
            if key not in self.folder_keys:
                npz = self.path / f'{key}.npz'
                if npz.exists():
                    self._cache[key] = np.load(npz)['arr_0']
                    logging.info(f"Cached '{key}' from '{npz}'")
        super().__init__(lengths, offsets, frameskip, num_steps, transform)

    @property
    def column_names(self) -> list[str]:
        return self._keys

    def _load_file(self, ep_idx: int, step: int, key: str) -> np.ndarray:
        "The image of `key` at a step: ``(H, W, C)``, or per agent ``(num_agents, H, W, C)``."
        base = self.path / key / f'ep_{ep_idx}_step_{step}'
        A = self._agents.get(key)
        if A is None:
            return _read_image(base)
        return np.stack([_read_image(base.with_name(f'{base.name}_agent_{a}')) for a in range(A)])

    def _load_slice(self, ep_idx: int, start: int, end: int) -> dict:
        g_start, g_end = self.offsets[ep_idx] + start, self.offsets[ep_idx] + end
        steps = {}
        for col in self._keys:
            if col in self.folder_keys:
                data = np.stack([self._load_file(ep_idx, s, col) for s in range(start, end, self.frameskip)])
            else:
                data = self._cache[col][g_start:g_end]
                if col != 'action':
                    data = data[::self.frameskip]
            if data.dtype == np.object_ or data.dtype.kind in ('S', 'U'):
                val = data[0] if len(data) > 0 else b''
                steps[col] = val.decode() if isinstance(val, bytes) else val
            else:
                steps[col] = to_tensor(data)
        return self.transform(steps) if self.transform else steps

    def get_col_data(self, col: str) -> np.ndarray:
        if col not in self._cache:
            raise KeyError(f"'{col}' not in cache (folder keys cannot be retrieved as full array)")
        return self._cache[col]

    def get_row_data(self, row_idx: int | list[int]) -> dict:
        return {c: self._cache[c][row_idx] for c in self._keys if c in self._cache}


class ImageDataset(FolderDataset):
    "``FolderDataset`` with ``pixels`` as the image folder."
    def __init__(self, name: str | None = None, image_keys: list[str] | None = None, **kw: Any):
        super().__init__(name=name, folder_keys=image_keys or ['pixels'], **kw)

In [ ]:
#| export
class _NpzEpisodeWriter:
    "Shared by the folder and video writers: `.npz` columns, episode index, modes, metadata."
    label = 'Writer'
    image_suffix = '.jpeg'

    def __init__(self, path, *, mode: str = 'append'):
        validate_write_mode(mode)
        self.path, self.mode = Path(path), mode
        self.path.mkdir(parents=True, exist_ok=True)
        self.metadata: dict[str, Any] = {}
        self._tabular: dict[str, list[np.ndarray]] = {}
        self._image_cols: set[str] = set()
        self._tabular_dims: dict[str, tuple[int, ...]] = {}
        self._lengths: list[int] = []
        self._offsets: list[int] = []
        self._global_ptr = self._ep_idx = 0
        self._appending_existing = self._validated = self._warned_episode_data = False

    def __enter__(self):
        if (self.path / 'ep_len.npz').exists():
            if self.mode == 'error':
                raise FileExistsError(f"{self.label}: '{self.path}' already contains a dataset. Pass mode='overwrite' "
                                      "to replace it or mode='append' to extend it.")
            if self.mode == 'overwrite':
                self._clear_existing()
            else:
                self._load_existing_state()
        return self

    def __exit__(self, *exc):
        self.close()

    def close(self):
        np.savez(self.path / 'ep_len.npz', np.asarray(self._lengths, np.int32))
        np.savez(self.path / 'ep_offset.npz', np.asarray(self._offsets, np.int64))
        for col, parts in self._tabular.items():
            np.savez(self.path / f'{col}.npz', np.concatenate(parts, axis=0))
        if self.metadata:
            (self.path / 'metadata.json').write_text(json.dumps(self.metadata, default=str))

    def write_episode(self, ep_data: dict) -> None:
        ep_data, extra = split_episode_data(ep_data)
        if extra and not self._warned_episode_data:
            logging.warning(f"{self.label}: episode data {sorted(extra)} is not stored in '{self.path}'.")
            self._warned_episode_data = True
        if self._appending_existing and not self._validated:
            self._validate_episode_against_existing(ep_data)
            self._validated = True
        ep_len = len(next(iter(ep_data.values())))
        for col, vals in ep_data.items():
            layout = image_layout(vals)
            if layout is not None:
                col_dir = self.path / col
                col_dir.mkdir(exist_ok=True)
                self._write_images(col_dir, np.asarray(vals), layout)
            else:
                self._tabular.setdefault(col, []).append(np.asarray(vals))
        self._lengths.append(ep_len)
        self._offsets.append(self._global_ptr)
        self._global_ptr += ep_len
        self._ep_idx += 1

    def write_episodes(self, episodes) -> None:
        for ep in episodes:
            self.write_episode(ep)

    def _write_images(self, col_dir: Path, frames: np.ndarray, layout: str):
        raise NotImplementedError

    def _is_image_dir(self, sub: Path) -> bool:
        return sub.is_dir()

    def _clear_existing(self) -> None:
        for f in ('ep_len.npz', 'ep_offset.npz', 'metadata.json'):
            (self.path / f).unlink(missing_ok=True)
        for child in self.path.iterdir():
            if child.is_file() and child.suffix == '.npz':
                child.unlink()
            elif child.is_dir():
                shutil.rmtree(child)

    def _load_existing_state(self) -> None:
        lengths = np.load(self.path / 'ep_len.npz')['arr_0']
        offsets = np.load(self.path / 'ep_offset.npz')['arr_0']
        self._lengths, self._offsets = lengths.astype(np.int32).tolist(), offsets.astype(np.int64).tolist()
        self._ep_idx = len(self._lengths)
        self._global_ptr = int(lengths.sum()) if len(lengths) else 0
        for npz in self.path.glob('*.npz'):
            if npz.stem in ('ep_len', 'ep_offset'):
                continue
            arr = np.load(npz)['arr_0']
            self._tabular[npz.stem] = [arr]
            self._tabular_dims[npz.stem] = tuple(arr.shape[1:])
        self._image_cols = {sub.name for sub in self.path.iterdir() if self._is_image_dir(sub)}
        meta = self.path / 'metadata.json'
        if meta.exists():
            self.metadata = {**json.loads(meta.read_text()), **self.metadata}
        self._appending_existing = True

    def _validate_episode_against_existing(self, ep_data: dict) -> None:
        incoming_image, incoming_tabular = set(), {}
        for col, vals in ep_data.items():
            if is_image_column(vals):
                incoming_image.add(col)
            else:
                incoming_tabular[col] = np.asarray(vals[0]).shape
        expected = self._image_cols | set(self._tabular_dims)
        incoming = incoming_image | set(incoming_tabular)
        missing, extra = expected - incoming, incoming - expected
        if missing or extra:
            raise ValueError(f"{self.label}: append failed — schema mismatch on '{self.path}'. "
                             f"Missing columns: {sorted(missing)}; unexpected columns: {sorted(extra)}.")
        bad_type = (incoming_image & set(self._tabular_dims)) | (set(incoming_tabular) & self._image_cols)
        if bad_type:
            raise ValueError(f"{self.label}: append failed — image-vs-tabular type mismatch for columns: {sorted(bad_type)}.")
        for col, shape in incoming_tabular.items():
            if shape != self._tabular_dims[col]:
                raise ValueError(f"{self.label}: append failed — column '{col}' per-step shape mismatch: "
                                 f"existing={self._tabular_dims[col]}, incoming={shape}.")


class FolderWriter(_NpzEpisodeWriter):
    """
    Appends episodes to a folder dataset (layout above): image columns as one JPEG per step (and
    per agent). `mode`: ``'append'`` (default), ``'overwrite'`` or ``'error'`` if a dataset is there.
    """
    label = 'FolderWriter'

    def _write_images(self, col_dir: Path, frames: np.ndarray, layout: str):
        for step, frame in enumerate(frames):
            if layout == 'single':
                _write_image(frame, col_dir / f'ep_{self._ep_idx}_step_{step}.jpeg')
            else:
                for a, agent_frame in enumerate(frame):
                    _write_image(agent_frame, col_dir / f'ep_{self._ep_idx}_step_{step}_agent_{a}.jpeg')


@register_format
class Folder(Format):
    "A folder with ``ep_len.npz`` (and no ``.mp4``: that is the video format)."
    name = 'folder'

    @classmethod
    def detect(cls, path) -> bool:
        p = Path(path)
        if not p.is_dir() or not (p / 'ep_len.npz').exists():
            return False
        return not any(sub.is_dir() and any(sub.glob('*.mp4')) for sub in p.iterdir())

    @classmethod
    def open_reader(cls, path, **kwargs) -> FolderDataset:
        return FolderDataset(path=path, **kwargs)

    @classmethod
    def open_writer(cls, path, **kwargs) -> FolderWriter:
        return FolderWriter(path, **kwargs)

### Tests

In [ ]:
import tempfile, torch
from fastcore.test import test_fail
import stable_marl as sm
from stable_marl.data import HDF5Dataset, detect_format, load_dataset, convert

ENV = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=8, max_episode_steps=6)
with tempfile.TemporaryDirectory() as tmp:
    world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **ENV)
    world.set_policy(sm.RandomPolicy(seed=0))
    world.collect(f'{tmp}/folder', episodes=3, seed=0, format='folder', progress=False)
    assert detect_format(f'{tmp}/folder') is Folder
    files = sorted(p.name for p in Path(f'{tmp}/folder/pixels').iterdir())
    assert files[0] == 'ep_0_step_0_agent_0.jpeg' and (Path(tmp) / 'folder' / 'goal').is_dir()   # per-agent images
    ds = load_dataset(f'{tmp}/folder', num_steps=3)
    assert ds.metadata['num_agents'] == 2 and set(ds.folder_keys) == {'pixels', 'goal'}
    item = ds[0]
    assert item['pixels'].shape == (3, 2, 56, 56, 3) and item['action'].shape == (3, 2) and item['position'].shape == (3, 2, 2)

    # same episodes through HDF5: tabular columns identical, images close (JPEG)
    world.set_policy(sm.RandomPolicy(seed=0))
    world.collect(f'{tmp}/d.h5', episodes=3, seed=0, progress=False)
    h5 = HDF5Dataset(path=f'{tmp}/d.h5', num_steps=3)
    a, b = ds[4], h5[4]
    assert torch.equal(a['position'], b['position']) and torch.allclose(a['action'], b['action'], equal_nan=True)
    assert (a['pixels'].float() - b['pixels'].float()).abs().mean() < 6
    # converting HDF5 -> folder, and appending to it
    convert(f'{tmp}/d.h5', f'{tmp}/converted', dest_format='folder', progress=False)
    assert load_dataset(f'{tmp}/converted').lengths.tolist() == h5.lengths.tolist()
    with FolderWriter(f'{tmp}/converted') as w:
        w.write_episode({c: list(h5.get_col_data(c)[:4]) for c in h5.column_names})
    assert len(load_dataset(f'{tmp}/converted').lengths) == 4
    test_fail(lambda: FolderWriter(f'{tmp}/converted', mode='error').__enter__(), contains='already contains')
    h5.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()